# 分组聚合：groupby

SQL 里 `GROUP BY` 在 pandas 的对应物。核心三步骤（split-apply-combine）：

1. **split** - 按某个键把数据拆成多个小组
2. **apply** - 对每个组执行聚合/变换/过滤
3. **combine** - 把各组结果拼回一张表

```python
df.groupby('列名').聚合函数()
df.groupby('列名').agg({'A': 'sum', 'B': 'mean'})
```

In [ ]:
import pandas as pd
import numpy as np
sales = pd.read_csv(r'D:/dsml/data/sales.csv')
students = pd.read_csv(r'D:/dsml/data/students.csv')
employees = pd.read_csv(r'D:/dsml/data/employees.csv')
print('sales:', sales.shape)
print('students:', students.shape)
print('employees:', employees.shape)


## 基础分组聚合

`groupby('列名')` 返回的不是 DataFrame，而是一个**分组对象**。接上聚合函数后才有结果。

```python
df.groupby('区域')['总金额'].sum()               # 按区域 -> 总销售额
df.groupby('区域')['总金额'].mean()              # 按区域 -> 平均订单额
df.groupby('区域')['订单ID'].count()             # 按区域 -> 订单数
```

**常见聚合函数**：`sum`, `mean`, `count`, `min`, `max`, `std`, `var`, `median`, `first`, `last`, `nunique`。

In [ ]:
# ===== 基础 groupby =====
print('--- 按区域 ---')
print(sales.groupby('区域')['总金额'].sum())

print('\n--- 按品类统计 ---')
print(sales.groupby('品类').agg({
    '订单ID': 'count',
    '总金额': 'sum',
    '单价': 'mean'
}))

print('\n--- 各品类最高单价 ---')
print(sales.groupby('品类')['单价'].max())

# as_index=False 可以保留原本索引

## agg() 的多种写法

### 方式 1：字典 - 每列不同函数
```python
df.groupby('区域').agg({'总金额': 'sum', '订单ID': 'count'})
```

### 方式 2：列表 - 每列相同函数组
```python
df.groupby('区域')['总金额'].agg(['sum', 'mean', 'std'])
```

### 方式 3：命名聚合（推荐，语义最清晰）
```python
df.groupby('区域').agg(
    总销售额=('总金额', 'sum'),
    订单数=('订单ID', 'count')
)
```

In [ ]:
# ===== agg 多种写法对比 =====
grp = sales.groupby('区域') #封存

print('--- 方式1: 字典 ---')
print(grp.agg({'总金额': 'sum', '订单ID': 'count'}))

print('\n--- 方式2: 列表 ---')
print(grp['总金额'].agg(['sum', 'mean', 'std']))
#外层列头是 '总金额'，内层是 ['sum', 'mean', 'std'] 访问用 result[('总金额', 'sum')] 或 result.loc[:, ('总金额', ['sum', 'mean'])] (与多级索引不一样)
# result.iloc[:, [0, 2]] 表示 sum 和 std 两列，如果第二列里面又分3个列，那第三列就要到4
print('\n--- 方式3: 命名聚合（推荐）---')
result = grp.agg(
    总销售额=('总金额', 'sum'),
    平均订单=('总金额', 'mean'),
    订单数=('订单ID', 'nunique')
)
#字典 + 列表会炸出多级列头，命名聚合产出的列头始终是你自己起的名字，绝不多级。之后 df['总销售额'] 一把能取出来，不需要元组。
print(result)


## transform() - 保持形状的聚合

`agg()` 把多行压缩成一行（每组一个值），`transform()` 把聚合结果**广播回每一行**——保持原 DataFrame 的行数不变。

```python
df['区域_均价'] = df.groupby('区域')['总金额'].transform('mean')
df['高于均价?'] = df['总金额'] > df['区域_均价']
```

**什么时候用**：你要在原表基础上加一列（如组内均值、组内排名），而不是要做汇总表。

| 方法 | 输出形状 | 用途 |
|---|---|---|
| `agg()` | 每组一行 | 生成汇总表 |
| `transform()` | 和原表一样行数 | 在原表上加列 |
| `filter()` | 原表的子集 | 过滤掉某些组 |
| `apply()` | 任意 | 自定义复杂操作 |

In [ ]:
# ===== transform 保持行数 =====
s = sales.copy()
s['区域_平均'] = s.groupby('区域')['总金额'].transform('mean') #transform也是按分类的样本空间计算，但是每一个类里面的不同记录不会被压缩成一条
#不限制一次操作一列，可以 df[['区域总金额', '区域均价']] = df.groupby('区域')[['总金额', '单价']].transform('mean')
s['高于区域平均'] = s['总金额'] > s['区域_平均']

print('每行都知道自己所在区域的平均:')
print(s[['区域', '总金额', '区域_平均', '高于区域平均']].head(10))

# 组内排名
s['区域内排名'] = s.groupby('区域')['总金额'].transform('rank', ascending=False)
# rank 是给每行贴一个"你是第几名"的标签，行位置不动
print('\n每个区域内的排名:')
print(s[['区域', '总金额', '区域内排名']].head(6))


## filter() - 过滤掉某些组

不是过滤行，是**过滤组**。传入的函数对整个组做判断，返回 True/False，留 True 的组。

```python
# 只保留订单数 >= 30 的区域
df.groupby('区域').filter(lambda g: len(g) >= 30)

# 只保留总销售额超过 10 万的品类
df.groupby('品类').filter(lambda g: g['总金额'].sum() > 100000)
```

In [ ]:
# ===== filter 过滤组 =====
print('各区域订单数:')
print(sales['区域'].value_counts())

big = sales.groupby('区域').filter(lambda g: len(g) >= 80)
#g 是区域的一个子 DataFrame，每一行就是一笔订单。len(g) 直接数这个区域有多少行
print('\n过滤后只保留大区域:', big['区域'].value_counts().index.tolist())
# .value_counts().index 等价于 .unique()，因为 .value_counts() 将传入列变为索引 索引天然去重
high = sales.groupby('品类').filter(lambda g: g['总金额'].sum() > 80000)
print('\n高销售额品类:', high['品类'].unique().tolist())
# filter 只是把不满足要求的品类全部删除，但是留下来的品类记录全保留，不会合并

## 多列分组

传入列表就给多个分组键，结果带多级行索引。

```python
df.groupby(['区域', '品类']).sum()                     # 区域 x 品类 的所有组合
df.groupby(['区域', '品类'])['总金额'].sum()            # 只看一列
df.groupby(['区域', '品类']).agg({'总金额': 'sum', '订单ID': 'count'})
```

**as_index=False**：结果变回普通 DataFrame，不把分组键变成索引。

In [ ]:
# ===== 多列分组 =====
print('--- 区域 x 品类 交叉统计 ---')
cross = sales.groupby(['区域', '品类']).agg(
    总销售额=('总金额', 'sum'),
    订单数=('订单ID', 'count')
).head(8)
print(cross)

flat = sales.groupby(['区域', '品类'], as_index=False)['总金额'].sum()
print('\nas_index=False:')
print(flat.head(6))

## apply() - 万能的组级操作

当 `agg` 内置函数不够用时，`apply` 可以给每个组执行任意自定义函数。

```python
# 每组 Top 3 大订单
df.groupby('区域').apply(lambda g: g.nlargest(3, '总金额'))

# 每组采样 2 行
df.groupby('品类').apply(lambda g: g.sample(2))
```

> `apply` 很灵活但比 `agg`/`transform` 慢。能用内置函数的就别上 `apply`。

In [ ]:
# ===== apply 自定义操作 =====
# groupby 后立即选列：分组键 '品类' 不进入 apply，避免 DeprecationWarning
print('--- 每个品类 Top 3 大订单 ---')
top3 = sales.groupby('品类')[['订单ID', '客户ID', '总金额']].apply(
    lambda g: g.nlargest(3, columns='总金额')
)
print(top3)

print('\n--- 每个区域随机抽 2 单 ---')
sample = sales.groupby('区域')[['订单ID', '总金额']].apply(
    lambda g: g.sample(2, random_state=42)
)
# random_state 是随机种子，使得每次抽到的两个样本一致
print(sample)


## agg vs apply 的区别

两者都能接收自定义函数对每个组做计算，但返回值类型不同时行为分裂：

| 返回值 | `agg()` | `apply()` |
|---|---|---|
| 标量（`max`、`mean`） | 行为一致：每个组一个值 | 行为一致：每个组一个值 |
| 多值（max+min） | **命名聚合**：`agg(A=('col','max'), B=('col','min'))` → 平级多列 ✓ | `apply` 返回 Series → keys 进索引内层，需 `unstack()` 才能变列 |
| 多行（Top 3） | ❌ 直接报错 | ✓ 每个组返回多行，自然拼接 |

**结论**：
- 标量聚合 → 用 `agg`（更快更安全）
- 多值聚合 → 用 `agg` 命名聚合（一行搞定，不绕路）
- 多行输出 → 只能用 `apply`
- `apply` 兜底一切，但有性能代价

In [ ]:
# ===== agg vs apply：关键区别 =====
import pandas as pd
sales = pd.read_csv(r'D:/dsml/data/sales.csv')

# agg：多值用命名聚合 —— 每组一行，结果平级多列
result_agg = sales.groupby('区域').agg(
    max_price=('总金额', 'max'),
    min_price=('总金额', 'min')
)
print('agg 命名聚合:')
print('列名:', result_agg.columns.tolist())
print(result_agg)

# apply：同样的需求也能做，但 keys 会进索引内层，需 unstack 才能变列
tmp = sales.groupby('区域')['总金额'].apply(
    lambda x: pd.Series({'max_price': x.max(), 'min_price': x.min()})
)
print('\napply 直接返回（看 index.names —— keys 在索引里）:')
print(f'index names: {tmp.index.names}')
print(tmp.head(8))

result_apply = tmp.unstack()
print('\nunstack 后（变回平级列）:')
print(result_apply)

# --- 真正的分水岭：多行输出 ---
print('\n--- agg 做不到、只有 apply 能扛的场景 ---')
top2 = sales.groupby('区域')[['订单ID', '总金额']].apply(
    lambda g: g.nlargest(2, '总金额')
)
print(top2)

print('总结：多行 → apply。多值 → agg 命名聚合最直接。')


## 综合练习

用 employees 表实战：按部门分组，统计薪资的均值/中位数/标准差，标出每部门的 Top 3 高薪。

In [ ]:
# ===== 综合：部门薪资分析 =====
e = employees.copy()

dept_stats = e.groupby('部门').agg(
    人数=('员工ID', 'count'),
    平均薪资=('月薪', 'mean'),
    中位数薪资=('月薪', 'median'),
    最高薪=('月薪', 'max'),
    最低薪=('月薪', 'min')
).round(2)
print('部门薪资统计:')
print(dept_stats)

# apply 前先选列，部门不进入 lambda，不会报警
top3 = e.groupby('部门')[['姓名', '月薪']].apply(lambda g: g.nlargest(3, '月薪'))
print('\n各月薪 Top 3:')
print(top3)

e['部门内排名'] = e.groupby('部门')['月薪'].transform('rank', ascending=False)
print('\n每人的部门内排名（前10行）:')
print(e[['姓名', '部门', '月薪', '部门内排名']].head(10))
